In [21]:
# --- Notebook bootstrap: make repo imports work ---
import sys
from pathlib import Path

nb_dir = Path.cwd()
repo_root = nb_dir.parent
sys.path.insert(0, str(repo_root))

print("Notebook dir:", nb_dir)
print("Repo root:", repo_root)

import re
import json
import numpy as np
import pandas as pd

from PIL import Image

from src.scene.roi import load_roi_config, ROIMaskEngine


Notebook dir: c:\Users\78222\Desktop\Bicylist-System\bicyclist-system\notebooks
Repo root: c:\Users\78222\Desktop\Bicylist-System\bicyclist-system


In [23]:
# --- User config ---
DATA_ROOT = Path(r"C:\Users\78222\Desktop\28_locations\0_MAIN_BIKE_DATASETS_clean")

LOC_ID = "loc_01"
IMG_DIR = DATA_ROOT / "Loc_01" / "Bicyclist"
ROI_JSON = repo_root / "configs" / "locations" / f"{LOC_ID}.json"

# Event grouping:
# Since filenames are continuous and camera triggers short bursts,
# we group images into events by numeric gaps in filenames.
EVENT_GAP = 2  # if image number jumps > EVENT_GAP, start a new event

# Detection settings
USE_YOLO = True
YOLO_MODEL = "yolov8n.pt"
TARGET_CLASS_IDS = {1}  # COCO bicycle=1 (can expand later)

# Keep a small confidence threshold if detections are sparse
CONF_MIN = 0.1

# Optional: limit number of images for quick debug
MAX_IMAGES = None

print("LOC_ID:", LOC_ID)
print("IMG_DIR:", IMG_DIR)
print("ROI_JSON:", ROI_JSON)
print("EVENT_GAP:", EVENT_GAP)


LOC_ID: loc_01
IMG_DIR: C:\Users\78222\Desktop\28_locations\0_MAIN_BIKE_DATASETS_clean\Loc_01\Bicyclist
ROI_JSON: c:\Users\78222\Desktop\Bicylist-System\bicyclist-system\configs\locations\loc_01.json
EVENT_GAP: 2


In [24]:
# --- Load ROI configuration ---
cfg = load_roi_config(ROI_JSON)
roi = ROIMaskEngine(cfg)

print("location_id:", cfg.location_id)
print("roi json image_size:", (cfg.w, cfg.h))
print("num polygons per roi:", {k: len(v) for k, v in cfg.rois.items()})
print("flow_vector:", roi.flow_vector())

# --- Collect images ---
img_paths = sorted(list(IMG_DIR.glob("*.JPG")))
if not img_paths:
    img_paths = sorted(list(IMG_DIR.glob("*.jpg")))
assert len(img_paths) > 0, f"No images found in {IMG_DIR}"

if MAX_IMAGES is not None:
    img_paths = img_paths[:MAX_IMAGES]

print("num images:", len(img_paths))
print("first/last:", img_paths[0].name, img_paths[-1].name)

img0 = Image.open(img_paths[0])
real_w, real_h = img0.size
print("real image_size:", img0.size, "sample:", img_paths[0].name)

need_scale = (real_w, real_h) != (cfg.w, cfg.h)
if need_scale:
    sx = cfg.w / real_w
    sy = cfg.h / real_h
    print("WARNING: real image size != ROI json size. Will scale detections by:", (sx, sy))
else:
    print("Image size matches ROI config. No scaling needed.")


location_id: loc_01
roi json image_size: (1920, 1088)
num polygons per roi: {'sidewalk': 0, 'bike_lane': 0, 'roadway': 1, 'crosswalk': 1, 'ignore_zone': 0}
flow_vector: None
num images: 224
first/last: IM_00165.JPG IM_04670.JPG
real image_size: (1920, 1088) sample: IM_00165.JPG
Image size matches ROI config. No scaling needed.


In [25]:
# --- Event grouping utilities (burst-level -> super-event / passage-level) ---

import re
from pathlib import Path

def extract_img_number(name: str) -> int:
    """
    Extract numeric id from filenames like IM_00340.JPG.
    Used as a proxy for temporal ordering in camera-trap data.
    """
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else -1


def group_images_into_events(img_paths, gap=2):
    """
    Group images into burst-level events using numeric gaps in filenames.

    Rule:
      - If the numeric gap between consecutive images <= gap,
        they are treated as belonging to the same trigger burst.
      - Otherwise, start a new burst event.

    Returns:
      events: list[list[Path]]
    """
    imgs = sorted(img_paths, key=lambda p: extract_img_number(p.name))
    if len(imgs) == 0:
        return []

    events = []
    cur = [imgs[0]]

    for prev, now in zip(imgs, imgs[1:]):
        prev_id = extract_img_number(prev.name)
        now_id  = extract_img_number(now.name)

        if (now_id - prev_id) <= gap:
            # Same trigger burst
            cur.append(now)
        else:
            # New trigger burst
            events.append(cur)
            cur = [now]

    events.append(cur)
    return events


def merge_events_into_super_events(events, super_gap=30):
    """
    Merge burst-level events into super-events (passage-level).

    Motivation:
      A single bicyclist may trigger the camera multiple times during one passage.
      Super-events reduce repeated counting by merging nearby bursts.

    Rule:
      - If the numeric gap between the end of one burst event and
        the start of the next burst event <= super_gap,
        merge them into the same super-event.

    Returns:
      super_events: list[list[Path]]
    """
    if len(events) == 0:
        return []

    def num(p: Path) -> int:
        return extract_img_number(p.name)

    super_events = []
    cur = list(events[0])

    for prev_ev, next_ev in zip(events, events[1:]):
        prev_last = num(prev_ev[-1])
        next_first = num(next_ev[0])

        if (next_first - prev_last) <= super_gap:
            # Same passage -> merge bursts
            cur.extend(next_ev)
        else:
            # New passage
            super_events.append(cur)
            cur = list(next_ev)

    super_events.append(cur)
    return super_events


# ------------------------------------------------------------------
# Apply grouping
# ------------------------------------------------------------------

EVENT_GAP = 2     # burst-level gap (2-second trigger)
SUPER_GAP = 7  # passage-level gap (tune to reduce repeated counting)

events = group_images_into_events(img_paths, gap=EVENT_GAP)
super_events = merge_events_into_super_events(events, super_gap=SUPER_GAP)

print("num burst events:", len(events))
print("num super-events:", len(super_events))

print("first 5 burst event sizes:", [len(e) for e in events[:5]])
print("first 5 super-event sizes:", [len(e) for e in super_events[:5]])


num burst events: 67
num super-events: 63
first 5 burst event sizes: [3, 11, 1, 2, 1]
first 5 super-event sizes: [3, 11, 1, 2, 1]


In [26]:
detections_df = pd.DataFrame(columns=["img","frame_global","x1","y1","x2","y2","score","cls"])

if USE_YOLO:
    from ultralytics import YOLO
    model = YOLO(YOLO_MODEL)

    rows = []
    for i, p in enumerate(img_paths):
        r = model(str(p), verbose=False)[0]
        if r.boxes is None:
            continue

        boxes = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        clss  = r.boxes.cls.cpu().numpy().astype(int)

        for (x1,y1,x2,y2), s, c in zip(boxes, confs, clss):
            if c not in TARGET_CLASS_IDS:
                continue
            if float(s) < CONF_MIN:
                continue
            rows.append({
                "img": p.name,
                "frame_global": int(i),
                "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2),
                "score": float(s),
                "cls": int(c),
            })

    detections_df = pd.DataFrame(rows)

# Scale detections into ROI coordinate system if needed
if len(detections_df) > 0 and need_scale:
    sx = cfg.w / real_w
    sy = cfg.h / real_h
    detections_df["x1"] *= sx
    detections_df["x2"] *= sx
    detections_df["y1"] *= sy
    detections_df["y2"] *= sy

print("detections_df:", detections_df.shape)
display(detections_df.head(10))


detections_df: (102, 8)


,img,frame_global,x1,y1,x2,y2,score,cls
0,IM_00216.JPG,3,1638.192627,537.276978,1803.196289,693.123291,0.828497,1
1,IM_00223.JPG,10,736.134766,512.523743,814.591309,624.631592,0.278081,1
2,IM_00365.JPG,15,891.256714,532.106567,1027.988159,626.132385,0.547178,1
3,IM_00619.JPG,22,777.631592,493.858154,989.930176,649.145325,0.764994,1
4,IM_00620.JPG,23,918.426819,506.525482,1070.467773,636.614380,0.509808,1
5,IM_00783.JPG,28,970.110596,517.957153,1136.838379,638.741211,0.518097,1
6,IM_00784.JPG,29,959.244690,503.195007,1078.813232,635.525452,0.255184,1
7,IM_00786.JPG,31,959.758850,508.793701,1078.687012,637.409729,0.314376,1
8,IM_00787.JPG,32,941.535522,512.643494,1076.414185,635.280029,0.331150,1
9,IM_00788.JPG,33,943.187256,518.415649,1077.109375,634.441895,0.372625,1


In [27]:
def bbox_center(bb):
    x1, y1, x2, y2 = bb
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

def bbox_area(bb):
    x1, y1, x2, y2 = bb
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def match_by_center_distance(
    boxes0,
    boxes1,
    max_dist=300.0,
    area_ratio_min=0.25,
    area_ratio_max=4.0,
):
    """
    Match one object between two frames by minimum center distance.
    More robust than IoU when displacement is large (IoU~0).

    Args:
      boxes0, boxes1: list of (x1,y1,x2,y2)
      max_dist: maximum allowed center distance (pixels in ROI coord system)
      area_ratio_min/max: reject pairs with big size mismatch

    Returns:
      (best_bb0, best_bb1, best_dist) or (None, None, inf)
    """
    if len(boxes0) == 0 or len(boxes1) == 0:
        return None, None, float("inf")

    best_bb0, best_bb1 = None, None
    best_d2 = float("inf")

    for bb0 in boxes0:
        a0 = bbox_area(bb0)
        c0x, c0y = bbox_center(bb0)

        for bb1 in boxes1:
            a1 = bbox_area(bb1)

            # Area ratio gate (prevents matching tiny far bike to a big near bike)
            if a0 > 1e-6 and a1 > 1e-6:
                ratio = a1 / a0
                if ratio < area_ratio_min or ratio > area_ratio_max:
                    continue

            c1x, c1y = bbox_center(bb1)
            d2 = (c1x - c0x) ** 2 + (c1y - c0y) ** 2
            if d2 < best_d2:
                best_d2 = d2
                best_bb0, best_bb1 = bb0, bb1

    if best_bb0 is None:
        return None, None, float("inf")

    # Distance gate
    if best_d2 > (max_dist ** 2):
        return None, None, float(best_d2) ** 0.5

    return best_bb0, best_bb1, float(best_d2) ** 0.5


In [28]:
def iou_xyxy(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 1e-6 else 0.0

def match_one_object_between_frames(boxes0, boxes1, min_iou=0.05):
    """
    For event-level wrong-way, we only need a coarse match between
    the first frame and the last frame in an event.

    Returns:
      (best_bb0, best_bb1, best_iou) or (None, None, 0)
    """
    if len(boxes0) == 0 or len(boxes1) == 0:
        return None, None, 0.0

    best = (None, None, 0.0)
    for bb0 in boxes0:
        for bb1 in boxes1:
            s = iou_xyxy(bb0, bb1)
            if s > best[2]:
                best = (bb0, bb1, s)

    if best[2] < min_iou:
        return None, None, best[2]
    return best


In [29]:
import pandas as pd
import numpy as np

# -------------------------
# Geometry helpers
# -------------------------
def bbox_bottom_center(bb):
    x1, y1, x2, y2 = bb
    return ((x1 + x2) / 2.0, y2)

def bbox_center(bb):
    x1, y1, x2, y2 = bb
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

# -------------------------
# Crosswalk touch (safe)
# -------------------------
def touch_crosswalk_for_bbox(bb, roi) -> bool:
    rois = getattr(roi.cfg, "rois", {})
    if (not rois.get("crosswalk")) or (len(rois.get("crosswalk", [])) == 0):
        return False

    x1, y1, x2, y2 = bb
    xs = [x1, (x1 + x2) / 2.0, x2]
    pts = [(float(x), float(y2)) for x in xs]
    return any(roi.point_in_roi(p, "crosswalk") for p in pts)

# -------------------------
# ROI gate (core fix)
# -------------------------
SPACE_ROIS = ("sidewalk", "bike_lane", "roadway")

def bbox_in_any_space_roi(bb, roi) -> bool:
    """Gate: bbox bottom-center must lie inside at least one space ROI."""
    bc = bbox_bottom_center(bb)
    for rn in SPACE_ROIS:
        if rn in roi.cfg.rois and len(roi.cfg.rois[rn]) > 0:
            if roi.point_in_roi(bc, rn):
                return True
    return False

def filter_detections_to_space_rois(det_df: pd.DataFrame, roi) -> pd.DataFrame:
    """Keep only detections whose bottom-center is inside any of the space ROIs."""
    if det_df is None or len(det_df) == 0:
        return det_df

    keep = []
    for _, r in det_df.iterrows():
        bb = (float(r.x1), float(r.y1), float(r.x2), float(r.y2))
        keep.append(bbox_in_any_space_roi(bb, roi))

    return det_df.loc[keep].copy()

# -------------------------
# Matching by center distance
# -------------------------
def match_by_center_distance(boxes0, boxes1, max_dist=400.0):
    if len(boxes0) == 0 or len(boxes1) == 0:
        return None, None, float("inf")

    best_bb0, best_bb1 = None, None
    best_d2 = float("inf")

    for bb0 in boxes0:
        c0x, c0y = bbox_center(bb0)
        for bb1 in boxes1:
            c1x, c1y = bbox_center(bb1)
            d2 = (c1x - c0x) ** 2 + (c1y - c0y) ** 2
            if d2 < best_d2:
                best_d2 = d2
                best_bb0, best_bb1 = bb0, bb1

    if best_bb0 is None:
        return None, None, float("inf")

    if best_d2 > (max_dist ** 2):
        return None, None, float(best_d2) ** 0.5

    return best_bb0, best_bb1, float(best_d2) ** 0.5

# -------------------------
# Event inference (with ROI gate)
# -------------------------
def infer_events(events, img_paths, detections_df, roi, min_move_px=8.0, max_match_dist=400.0):
    flow = roi.flow_vector()
    has_flow = flow is not None

    img2idx = {p.name: i for i, p in enumerate(img_paths)}
    rows = []

    for eid, ev_imgs in enumerate(events):
        ev_names = [p.name for p in ev_imgs]

        # 1) pull detections in this event
        det_ev = detections_df[detections_df["img"].isin(ev_names)].copy()

        # 2) ROI gate: remove detections outside sidewalk/bike_lane/roadway
        det_ev = filter_detections_to_space_rois(det_ev, roi)

        has_target = len(det_ev) > 0

        # 3) crosswalk touch (still safe even when no crosswalk ROI)
        touch_cw = False
        if has_target:
            for _, r in det_ev.iterrows():
                bb = (float(r.x1), float(r.y1), float(r.x2), float(r.y2))
                if touch_crosswalk_for_bbox(bb, roi):
                    touch_cw = True
                    break

        # 4) wrong-way
        wrong_way = pd.NA
        direction = pd.NA
        cos_to_flow = pd.NA

        if has_flow and has_target:
            det_ev2 = det_ev.copy()
            det_ev2["frame_global"] = det_ev2["img"].map(img2idx)
            det_ev2 = det_ev2.dropna(subset=["frame_global"])

            # Need at least two detected frames to estimate direction
            if len(det_ev2) >= 2:
                fg_min = int(det_ev2["frame_global"].min())
                fg_max = int(det_ev2["frame_global"].max())

                f0 = img_paths[fg_min].name
                f1 = img_paths[fg_max].name

                det0 = det_ev2[det_ev2["img"] == f0]
                det1 = det_ev2[det_ev2["img"] == f1]

                boxes0 = [(float(r.x1), float(r.y1), float(r.x2), float(r.y2)) for _, r in det0.iterrows()]
                boxes1 = [(float(r.x1), float(r.y1), float(r.x2), float(r.y2)) for _, r in det1.iterrows()]

                bb0, bb1, dist = match_by_center_distance(boxes0, boxes1, max_dist=max_match_dist)

                if bb0 is not None and bb1 is not None:
                    bc0 = bbox_bottom_center(bb0)
                    bc1 = bbox_bottom_center(bb1)
                    vx, vy = (bc1[0] - bc0[0]), (bc1[1] - bc0[1])
                    mag = float((vx * vx + vy * vy) ** 0.5)

                    if mag >= min_move_px:
                        ux, uy = vx / mag, vy / mag
                        cos = float(ux * float(flow[0]) + uy * float(flow[1]))
                        direction = "along_flow" if cos >= 0 else "against_flow"
                        cos_to_flow = cos
                        wrong_way = (direction == "against_flow")

        rows.append({
            "event_id": eid,
            "n_images": len(ev_imgs),
            "img_first": ev_names[0],
            "img_last": ev_names[-1],
            "has_target": bool(has_target),
            "touch_crosswalk": bool(touch_cw),
            "has_flow": bool(has_flow),
            "wrong_way": wrong_way,
            "direction": direction,
            "cos_to_flow": cos_to_flow,
        })

    return pd.DataFrame(rows)

# -------------------------
# RUN
# -------------------------
events_df = infer_events(
    events,
    img_paths,
    detections_df,
    roi,
    min_move_px=8.0,
    max_match_dist=400.0
)
print("events_df:", events_df.shape)
display(events_df.head(10))


events_df: (67, 10)


,event_id,n_images,img_first,img_last,has_target,touch_crosswalk,has_flow,wrong_way,direction,cos_to_flow
0,0,3,IM_00165.JPG,IM_00168.JPG,False,False,False,<NA>,<NA>,<NA>
1,1,11,IM_00216.JPG,IM_00226.JPG,False,False,False,<NA>,<NA>,<NA>
2,2,1,IM_00251.JPG,IM_00251.JPG,False,False,False,<NA>,<NA>,<NA>
3,3,2,IM_00365.JPG,IM_00366.JPG,False,False,False,<NA>,<NA>,<NA>
4,4,1,IM_00485.JPG,IM_00485.JPG,False,False,False,<NA>,<NA>,<NA>
5,5,2,IM_00519.JPG,IM_00520.JPG,False,False,False,<NA>,<NA>,<NA>
6,6,2,IM_00557.JPG,IM_00558.JPG,False,False,False,<NA>,<NA>,<NA>
7,7,2,IM_00619.JPG,IM_00620.JPG,False,False,False,<NA>,<NA>,<NA>
8,8,2,IM_00719.JPG,IM_00720.JPG,False,False,False,<NA>,<NA>,<NA>
9,9,11,IM_00781.JPG,IM_00791.JPG,True,True,False,<NA>,<NA>,<NA>


In [30]:
# --- Main metric: event-level crosswalk touch rate ---
mask_has = events_df["has_target"] == True
num_touch = int((events_df.loc[mask_has, "touch_crosswalk"] == True).sum())
den_touch = int(mask_has.sum())
touch_rate = (num_touch / den_touch) if den_touch > 0 else None

print("Event-level Crosswalk Touch Rate:")
print("  num_touch_crosswalk =", num_touch)
print("  denom_events_with_target =", den_touch)
print("  touch_rate =", touch_rate)

# --- Wrong-way: only count events where wrong_way is not NA ---
mask_app = events_df["wrong_way"].notna()
den_ww = int(mask_app.sum())
num_ww = int((events_df.loc[mask_app, "wrong_way"] == True).sum()) if den_ww > 0 else 0
ww_rate = (num_ww / den_ww) if den_ww > 0 else None

print("\nEvent-level Wrong-way Rate (applicable only):")
print("  applicable_events =", den_ww)
print("  wrong_way_true =", num_ww)
print("  wrong_way_rate =", ww_rate)


Event-level Crosswalk Touch Rate:
  num_touch_crosswalk = 8
  denom_events_with_target = 13
  touch_rate = 0.6153846153846154

Event-level Wrong-way Rate (applicable only):
  applicable_events = 0
  wrong_way_true = 0
  wrong_way_rate = None


In [31]:
outdir = repo_root / "outputs" / f"{LOC_ID}_eventA"
outdir.mkdir(parents=True, exist_ok=True)

detections_df.to_csv(outdir / "detections.csv", index=False)
events_df.to_csv(outdir / "events.csv", index=False, na_rep="NA")

summary = {
    "loc": LOC_ID,
    "roi_json": str(ROI_JSON),
    "img_dir": str(IMG_DIR),
    "n_images": len(img_paths),
    "n_events": int(len(events)),
    "detections_rows": int(len(detections_df)),
    "event_touch_rate": touch_rate,
    "event_wrong_way_rate_applicable_only": ww_rate,
    "event_wrong_way_applicable": den_ww,
}

(outdir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("[OK] saved to:", outdir)
summary


[OK] saved to: c:\Users\78222\Desktop\Bicylist-System\bicyclist-system\outputs\loc_01_eventA


{'loc': 'loc_01',
 'roi_json': 'c:\\Users\\78222\\Desktop\\Bicylist-System\\bicyclist-system\\configs\\locations\\loc_01.json',
 'img_dir': 'C:\\Users\\78222\\Desktop\\28_locations\\0_MAIN_BIKE_DATASETS_clean\\Loc_01\\Bicyclist',
 'n_images': 224,
 'n_events': 67,
 'detections_rows': 102,
 'event_touch_rate': 0.6153846153846154,
 'event_wrong_way_rate_applicable_only': None,
 'event_wrong_way_applicable': 0}